In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'robustness' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
scoredRows = predictions.join(evaluationRows.select('rowId', 'ndvi', 'flaggedObservation'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
allRows = scoredRows.with_columns(pl.lit('all').alias('sensitivitySet'))
unflaggedRows = scoredRows.filter(~pl.col('flaggedObservation')).with_columns(pl.lit('unflagged').alias('sensitivitySet'))
flaggedRows = scoredRows.filter(pl.col('flaggedObservation')).with_columns(pl.lit('flagged').alias('sensitivitySet'))
flaggedObservationSensitivity = pl.concat([allRows, unflaggedRows, flaggedRows]).group_by('sensitivitySet', 'model', 'featureSet', 'seed').agg(pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')).sort('sensitivitySet', 'model', 'featureSet', 'seed')
flaggedObservationSensitivity.write_parquet(outputPath / 'flaggedObservationSensitivity.parquet')
assert evaluationRows.filter(pl.col('flaggedObservation')).height == 14
assert flaggedObservationSensitivity.filter(pl.col('sensitivitySet') == 'flagged').select(pl.col('rowCount').unique()).get_column('rowCount').to_list() == [14]
flaggedObservationSensitivity
